# Imports

In [1]:
import optuna

import numpy as np
import pandas as pd

from sklearn.metrics import balanced_accuracy_score, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one.parquet') #.drop(to_drop, axis=1)
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one.parquet') #.drop(to_drop, axis=1)

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.004831,0.002142,0.993027,0.005220,0.002128,0.992652,0.005655,0.000463,0.993882
1,0.998229,0.000143,0.001628,0.997774,0.000322,0.001904,0.994638,0.000368,0.004994
2,0.003982,0.000584,0.995434,0.002362,0.000710,0.996928,0.002327,0.000919,0.996754
3,0.007424,0.001954,0.990622,0.002922,0.003628,0.993450,0.003433,0.002374,0.994194
4,0.993583,0.000012,0.006405,0.997151,0.000013,0.002836,0.994112,0.000005,0.005883


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.011986,0.002516,0.985498,0.009749,0.003956,0.986295,0.008185,0.005526,0.986288
1,0.469590,0.002922,0.527488,0.344531,0.002104,0.653365,0.530403,0.002094,0.467503
2,0.998492,0.000991,0.000517,0.998555,0.001132,0.000313,0.996306,0.003216,0.000478
3,0.986619,0.000009,0.013372,0.978274,0.000024,0.021702,0.990683,0.000093,0.009224
4,0.008520,0.001975,0.989505,0.005601,0.002080,0.992319,0.003507,0.001947,0.994546


# Machine Learning

In [ ]:
def objective(trial, X, y):
    
    l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)
    C = trial.suggest_float("C", 1e-5, 100.0, log=True)
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
    fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])
    tol = trial.suggest_float("tol", 1e-6, 1e-2, log=True)
    max_iter = trial.suggest_int("max_iter", 1000, 5000)

    w0 = trial.suggest_float('weight_class_0', 0.05, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.05, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.05, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        
        X_train_fold, X_valid_fold = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]

        model = LogisticRegression(
            solver="saga",
            C=C,
            l1_ratio=l1_ratio,
            class_weight=class_weight,
            fit_intercept=fit_intercept,
            tol=tol,
            max_iter=max_iter,
            random_state=42,
        ).fit(X_train_fold, y_train_fold)

        proba = model.predict_proba(X_valid_fold)
        
        weighted_probas = proba * weights
        pred = np.argmax(weighted_probas, axis=1)
        
        score = balanced_accuracy_score(y_valid_fold, pred)
        scores.append(score)

        trial.report(np.mean(scores), step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=30, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-05 15:50:45,094] A new study created in memory with name: no-name-77c8b126-b7a4-4ee9-8970-9b9c3f06edc0
Best trial: 6. Best value: 0.88965:   3%|████▋                                                                                                                                      | 1/30 [00:37<18:16, 37.80s/it]

[I 2026-07-05 15:51:22,878] Trial 6 finished with value: 0.8896502827517431 and parameters: {'l1_ratio': 0.8599735073948968, 'C': 0.061724338790983, 'class_weight': None, 'fit_intercept': False, 'tol': 0.0007242402287206063, 'max_iter': 2995, 'weight_class_0': 0.6354361120667588, 'weight_class_1': 7.150304299616766, 'weight_class_2': 0.06786627877696891}. Best is trial 6 with value: 0.8896502827517431.


Best trial: 6. Best value: 0.88965:   7%|█████████▎                                                                                                                                 | 2/30 [00:43<08:44, 18.75s/it]

[I 2026-07-05 15:51:28,300] Trial 8 finished with value: 0.8360329086730831 and parameters: {'l1_ratio': 0.1071608892526591, 'C': 26.16360974044426, 'class_weight': None, 'fit_intercept': True, 'tol': 0.0008706098904997742, 'max_iter': 1895, 'weight_class_0': 2.2576772773047344, 'weight_class_1': 0.1285033632156152, 'weight_class_2': 0.08240698902933564}. Best is trial 6 with value: 0.8896502827517431.


Best trial: 7. Best value: 0.893531:  10%|█████████████▊                                                                                                                            | 3/30 [00:45<05:03, 11.25s/it]

...

Best trial: 23. Best value: 0.942317:  53%|████████████████████████████████████████████████████████████████████████                                                               | 16/30 [11:34<32:42, 140.17s/it]

[I 2026-07-05 16:02:19,475] Trial 23 finished with value: 0.9423170317762566 and parameters: {'l1_ratio': 0.5553690006663523, 'C': 2.802165191305813, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 2.528569649012793e-06, 'max_iter': 1009, 'weight_class_0': 8.636094349122967, 'weight_class_1': 9.648346735539304, 'weight_class_2': 8.881410940043384}. Best is trial 23 with value: 0.9423170317762566.


/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Best trial: 23. Best value: 0.942317:  57%|████████████████████████████████████████████████████████████████████████████▌                                                          | 17/30 [12:42<25:40, 118.53s/it]

[I 2026-07-05 16:03:27,671] Trial 20 finished with value: 0.9188627680871294 and parameters: {'l1_ratio': 0.6153220664430329, 'C': 12.057013914550447, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.008734853727935976, 'max_iter': 4742, 'weight_class_0': 4.575941137803842, 'weight_class_1': 0.17805981938287996, 'weight_class_2': 5.0699507806402435}. Best is trial 23 with value: 0.9423170317762566.


/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [28]:
lg_params = {k: v for k, v in study.best_params.items() if k not in ['weight_class_0', 'weight_class_1', 'weight_class_2']}

lg = LogisticRegression(
    solver="saga",
    random_state=42,
    **lg_params
).fit(X_train, y_train.class_encoded)

test_proba = lg.predict_proba(X_test)

weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = test_proba * weights

pred = np.argmax(weighted_probas, axis=1)

In [29]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [30]:
submission = pd.read_csv('../data/sample_submission.csv')
submission['class'] = sub_labels

submission.to_csv('../data/submission_stacking_lg.csv', index=False)

In [31]:
submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [32]:
X_train.columns

Index(['lgbm_0', 'lgbm_1', 'lgbm_2', 'cat_0', 'cat_1', 'cat_2', 'xgb_0',
       'xgb_1', 'xgb_2', 'lg_0', 'lg_1', 'lg_2', 'sgd_0', 'sgd_1', 'sgd_2'],
      dtype='str')

In [33]:
len(study.trials)

200